In [2]:
import numpy as np
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, Slider, CustomJS, Label
from bokeh.plotting import figure, show, output_file
from bokeh.io import curdoc

# Başlangıç parametreleri
sigma = 10
beta = 8 / 3
rho = 28
dt = 0.01
num_steps = 10000

# Lorenz sisteminin başlangıç koşulları
x, y, z = 0.0, 1.0, 1.05

# Boş listeler veriyi saklamak için
xs, ys, zs = [], [], []

# Lorenz sistemi için fonksiyon
def lorenz_caner(sigma, beta, rho):
    global x, y, z
    x, y, z = 0.0, 1.0, 1.05  # Başlangıç koşullarını sıfırla
    xs, ys, zs = [], [], []
    for _ in range(num_steps):
        dx = sigma * (y - x) * dt
        dy = (x * (rho - z) - y) * dt
        dz = (x * y - beta * z) * dt
        x += dx
        y += dy
        z += dz
        xs.append(x)
        ys.append(y)
        zs.append(z)
    return xs, ys, zs

xs, ys, zs = lorenz_caner(sigma, beta, rho)

# Verileri ColumnDataSource'a koymak
source = ColumnDataSource(data=dict(x=xs, y=ys, z=zs))

# Grafik oluşturma
p = figure(width=800, height=800, title="Caner'in Lorenz Çekici")
p.line('x', 'y', source=source, line_width=2, line_alpha=0.7)

# Sağ alt köşeye isim ekleme
label = Label(x=70, y=70, x_units='screen', y_units='screen', text='Rahan Hoca',
              border_line_color='black', border_line_alpha=1.0,
              background_fill_color='white', background_fill_alpha=1.0)

p.add_layout(label)

# Sigma, Beta ve Rho kaydırıcıları
sigma_slider = Slider(start=0, end=50, value=sigma, step=0.1, title="Sigma")
beta_slider = Slider(start=0, end=10, value=beta, step=0.1, title="Beta")
rho_slider = Slider(start=0, end=50, value=rho, step=0.1, title="Rho")

# JavaScript callback fonksiyonu
callback = CustomJS(args=dict(source=source, sigma_slider=sigma_slider, beta_slider=beta_slider, rho_slider=rho_slider), code="""
    const data = source.data;
    const sigma = sigma_slider.value;
    const beta = beta_slider.value;
    const rho = rho_slider.value;
    const dt = 0.01;
    const num_steps = 10000;

    let x = 0.0;
    let y = 1.0;
    let z = 1.05;

    const xs = [];
    const ys = [];
    const zs = [];

    for (let i = 0; i < num_steps; i++) {
        const dx = sigma * (y - x) * dt;
        const dy = (x * (rho - z) - y) * dt;
        const dz = (x * y - beta * z) * dt;
        x += dx;
        y += dy;
        z += dz;
        xs.push(x);
        ys.push(y);
        zs.push(z);
    }

    data['x'] = xs;
    data['y'] = ys;
    data['z'] = zs;

    source.change.emit();
""")

sigma_slider.js_on_change('value', callback)
beta_slider.js_on_change('value', callback)
rho_slider.js_on_change('value', callback)

# Layout
layout = column(p, sigma_slider, beta_slider, rho_slider)

# Çıktı dosyasını belirleyip grafiği gösterelim
output_file("caner_lorenz_attractor_with_sliders.html")
show(layout)
curdoc().add_root(layout)


RuntimeError: Models must be owned by only a single document, LinearAxis(id='p1016', ...) is already in a doc